# Landing — S&P 500 tracker prices

Source: Yahoo Finance via the `yfinance` library — not an official API, but the
established free way to read the same data Yahoo's site shows.

Four tickers, all tracking the S&P 500 in different wrappers. **SPY is the
benchmark** used in every calculation (its history goes back to 1993); the other
three exist only for a secondary "these trackers should overlap" credibility chart.

Lands **every daily row the source holds**, as it arrives. Cutting the history to the
15-year horizon, and collapsing it to monthly, are both business rules and belong in
Silver.

In [0]:
%pip install yfinance

In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
import yfinance as yf

CATALOG = "`index-vs-trust-pipeline`"
TABLE = f"{CATALOG}.landing.index_prices_raw"

INDEX_TICKERS = ["SPY", "IVV", "VOO", "SPLG"]

In [0]:
pulls = []

for ticker in INDEX_TICKERS:
    print(f"downloading {ticker}...")

    # period="max" takes the whole history the source holds. Windowing it to the 15-year
    # horizon is a business rule and belongs in Silver.
    # auto_adjust=False keeps the raw Close separate from the adjusted one -- we need the
    # raw Close, because the trust data has no dividends to match against.
    hist = yf.Ticker(ticker).history(period="max", auto_adjust=False, actions=True)

    # Yahoo returns an empty frame for a ticker it no longer serves. Nothing arrived, so
    # there is nothing to land -- say so and move on. Landing records what the source
    # gave, and an absent source is not an error to correct here.
    if hist.empty:
        print("  -> nothing returned")
        continue

    # yfinance puts the date in the row label, not a column.
    hist = hist.reset_index()

    # Delta rejects spaces in column names, so "Adj Close" cannot be stored as it stands.
    # A rename, not a change of data.
    hist.columns = [c.replace(" ", "_") for c in hist.columns]

    # Spark cannot store a timezone-aware pandas timestamp, so the market timezone has to
    # come off to land the row at all. yfinance omits it on some responses, and stripping
    # an absent timezone raises.
    if hist["Date"].dt.tz is not None:
        hist["Date"] = hist["Date"].dt.tz_localize(None)

    # The response never says which ETF it is, so without this column the rows are
    # unusable. Identification, not transformation.
    hist.insert(0, "ticker", ticker)

    print(f"  -> {len(hist)} rows, {hist['Date'].min().date()} to {hist['Date'].max().date()}")
    pulls.append(hist)

index_prices = pd.concat(pulls, ignore_index=True)
print(f"total {len(index_prices)} rows across {index_prices['ticker'].nunique()} tickers")
index_prices.head(3)

In [0]:
sdf = spark.createDataFrame(index_prices)
sdf.write.format("delta").mode("overwrite").saveAsTable(TABLE)

print(f"wrote {TABLE}")

## Verification

In [0]:
%sql
SELECT ticker,
       COUNT(*) AS row_count,
       MIN(Date) AS first_date,
       MAX(Date) AS last_date
FROM `index-vs-trust-pipeline`.landing.index_prices_raw
GROUP BY ticker
ORDER BY ticker;

We ask Yahoo for four tickers and take each one's full history. Expect **3 in the
table**, each starting on its own launch date — the counts differ because the funds
are different ages, which is the data, not a bug:

| ticker | first date | approx. rows |
|---|---|---|
| SPY | 1993-01-29 | ~8,200 |
| IVV | 2000-05-19 | ~6,400 |
| VOO | 2010-09-09 | ~3,800 |

Row counts are approximate because they grow by one each trading day. The **first
dates are exact** and are the real check: if SPY starts in 2011 rather than 1993, the
history has been cut short somewhere.

**SPLG returns nothing.** Yahoo's API answers `No data found, symbol may be delisted`
for it, so zero SPLG rows land. That is the source, not a bug, and Landing records what
arrived rather than substituting a fix. SPLG only ever fed the secondary "trackers
overlap" chart; SPY, the benchmark every calculation uses, is unaffected.